In [2]:
import sys
!{sys.executable} -m pip install understatapi --quiet

In [3]:
import pandas as pd
import understatapi

client = understatapi.UnderstatClient()

In [4]:
# Let's first get data from a league to retrieve a list of matches
league_data = client.league(league="EPL").get_match_data(season="2024")
league_data[0]

{'id': '26602',
 'isResult': True,
 'h': {'id': '89', 'title': 'Manchester United', 'short_title': 'MUN'},
 'a': {'id': '228', 'title': 'Fulham', 'short_title': 'FLH'},
 'goals': {'h': '1', 'a': '0'},
 'xG': {'h': '2.04268', 'a': '0.418711'},
 'datetime': '2024-08-16 19:00:00',
 'forecast': {'w': '0.8069', 'd': '0.1489', 'l': '0.0442'}}

In [11]:
all_dfs = []
for season in ['2019', '2020', '2021', '2022', '2023', '2024', '2025']:
    print(f"Ophalen {season}/{int(season)+1}...")
    matches = client.league(league="EPL").get_match_data(season=season)
    skipped = 0
    for match in matches:
        # Sla wedstrijden over die nog niet gespeeld zijn
        if match['xG']['h'] is None or match['xG']['a'] is None:
            skipped += 1
            continue
        all_dfs.append({
            'date':       match['datetime'][:10],
            'home_team':  match['h']['title'],
            'away_team':  match['a']['title'],
            'home_xg':    float(match['xG']['h']),
            'away_xg':    float(match['xG']['a']),
            'home_goals': int(match['goals']['h']),
            'away_goals': int(match['goals']['a']),
            'season':     f"{season}/{int(season)+1}",
        })
    print(f"  → {len(matches)-skipped} gespeeld, {skipped} overgeslagen")

xg_all = pd.DataFrame(all_dfs)
xg_all.to_csv('understat_xg.csv', index=False)
print(f"\nKlaar! {len(xg_all)} wedstrijden opgeslagen")
xg_all.head()

Ophalen 2019/2020...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2020/2021...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2021/2022...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2022/2023...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2023/2024...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2024/2025...
  → 380 gespeeld, 0 overgeslagen
Ophalen 2025/2026...
  → 301 gespeeld, 79 overgeslagen

Klaar! 2581 wedstrijden opgeslagen


,date,home_team,away_team,home_xg,away_xg,home_goals,away_goals,season
0,2019-08-09,Liverpool,Norwich,2.234560,0.842407,4,1,2019/2020
1,2019-08-10,West Ham,Manchester City,1.200300,3.183770,0,5,2019/2020
2,2019-08-10,Bournemouth,Sheffield United,1.340990,1.598640,1,1,2019/2020
3,2019-08-10,Burnley,Southampton,0.909241,1.087520,3,0,2019/2020
4,2019-08-10,Crystal Palace,Everton,0.871590,1.224600,0,0,2019/2020
